In [1]:
import pandas as pd
import numpy as np

In [20]:
df_tr = pd.read_csv('D:\\repos\\ML_innovise\\data\\processed\\df_train_eda.csv')
df_test = pd.read_csv('D:\\repos\\ML_innovise\\data\\processed\\df_test_eda.csv')

In [21]:
df_testt = pd.read_csv(r'D:\ML_innovise\test.csv')
df_test = df_test.merge(
    df_testt,
    on = ['shop_id','item_id'],
    how = 'left'
)

In [4]:
from Splitter import Splitter

splitter = Splitter('date_block_num', 33)


In [22]:
features =['cnt_lag_1','cnt_lag_3','cnt_lag_2','cnt_lag_12', 'cnt_rm_3','cnt_rm_6','item_avg_price_prev', 'item_price','date_block_num','avg_price_prev_missing']
X_tr, y_tr, X_val, y_val = splitter.split(df_tr, features,'item_cnt_month')


In [30]:
from sklearn.ensemble import RandomForestRegressor


In [ ]:

modelRFR = RandomForestRegressor(n_estimators = 200, max_depth = 10, random_state= 42, n_jobs = -1)

In [7]:
modelRFR.fit(X_tr,y_tr)

,n_estimators,200
,criterion,'squared_error'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
imp = pd.Series(modelRFR.feature_importances_, index = X_tr.columns).sort_values(ascending=False)

In [9]:
imp

item_price                0.565264
cnt_lag_1                 0.324928
cnt_rm_6                  0.050793
item_avg_price_prev       0.024062
date_block_num            0.018359
cnt_lag_3                 0.009237
cnt_lag_2                 0.005839
cnt_lag_12                0.001424
avg_price_prev_missing    0.000095
dtype: float64

item_price + cnt_lag_1 дают почти 90 процентов важности

Интересный факт: цена имеет наибольшую важность, но корреляция у цены с таргетом маленькая (0.35)

Значит скорее всего присутствует какая-то коварная нелинейная связь между ними.

In [10]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(modelRFR, X_val, y_val, n_repeats = 10, random_state = 42, n_jobs = -1)
 

In [11]:
perm_imp = pd.Series(perm.importances_mean, index = X_val.columns).sort_values(ascending=False)
perm_imp 

item_price                0.865597
cnt_lag_1                 0.109554
cnt_rm_6                  0.051026
item_avg_price_prev       0.024206
cnt_lag_2                 0.003526
cnt_lag_3                 0.003377
cnt_lag_12                0.002025
avg_price_prev_missing    0.000003
date_block_num            0.000000
dtype: float64

видим, что  после перестановки топовые лидеры по важности: item_price   cnt_lag_1   cnt_rm_6

In [6]:
from hyperopt import hp, Trials, fmin, tpe, STATUS_OK

d:\repos\ML_innovise\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [7]:
from sklearn.model_selection import cross_val_score, TimeSeriesSplit

In [ ]:
space = {
    'n_estimators': hp.choice('n_estimators', [50, 100] ),
    'max_depth': hp.choice('max_depth', [5,10]),
    'min_samples_split': hp.quniform('min_samples_split', 2,5,1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 3, 1),
}

splitter = TimeSeriesSplit(n_splits = 5)

def objective(params):
    params['min_samples_leaf'] = int(params['min_samples_leaf'])
    params['min_samples_split'] = int(params['min_samples_split'])

    model = RandomForestRegressor(
        n_estimators = params['n_estimators'],
        max_depth = params['max_depth'],
        min_samples_split = params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        random_state=42
        )
    metric = cross_val_score(model, X_tr, y_tr, cv = splitter, scoring = 'neg_mean_squared_error', n_jobs = -1).mean()
    return {
        'loss': -metric,
        'status': STATUS_OK
    }

trials = Trials()
best = fmin(
    fn = objective,
    space = space,
    algo = tpe.suggest,
    max_evals = 30,
    trials = trials,
)

print(best)



 73%|███████▎  | 22/30 [1:29:32<30:30, 228.82s/trial, best loss: 0.4027374525028648] 

In [8]:
from sklearn.linear_model import Ridge

In [23]:
space = {
    'alpha': hp.loguniform('alpha', np.log(1e-4), np.log(1e2))
}
tscv = TimeSeriesSplit(n_splits=5)
def objective(params):
    model = Ridge(alpha = params['alpha'], random_state = 42)
    metric = cross_val_score(model, X_tr, y_tr, cv = tscv, scoring  = 'neg_mean_squared_error', n_jobs = -1).mean()
    return{
        'loss': -metric,
        'status': STATUS_OK
    }
trials = Trials()
best = fmin(fn = objective, space = space, algo = tpe.suggest, max_evals = 15, trials = trials, )
print('Best alpha:', best['alpha'])

100%|██████████| 15/15 [00:50<00:00,  3.40s/trial, best loss: 0.6061698965738761]
Best alpha: 0.00038128326956847947


In [26]:
alpha_opt = best['alpha']
model = Ridge(alpha = alpha_opt)
model.fit(X_tr, y_tr)
preds = model.predict(df_test[features])


In [27]:
submission = pd.DataFrame({
    "ID":              df_test["ID"],
    "item_cnt_month":  preds
})

In [28]:
submission.to_csv("RidgeAlphaOpt2.csv", index=False)

Результат - 1.03091